<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Fine_Tuning_LLMs_and_Inference_(vLLM)_with_an_RTX_5090.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Fine-Tuning LLMs and Inference (vLLM) with an RTX 5090](https://kaitchup.substack.com/p/fine-tuning-llms-and-inference-vllm)*

This notebook shows how to set up your environment to run vLLM and fine-tuning with an RTX 50xx. It has been tested with the RTX 5090 and 5080, and should also work with an RTX 5070.

You will need CUDA 12.8.



# Upgrade Your PyTorch

I had to run the following to upgrade PyTorch. Once the PyTorch 2.8 is the stable release, I would assume that you won't need to run this, but just a normal pip install --upgrade torch torchvision torchaudio.

In [ ]:
!pip3 install --upgrade --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

# Install vLLM

In [ ]:
!git clone https://github.com/vllm-project/vllm.git
%cd vllm
!python use_existing_torch.py
!pip install -r requirements/build.txt
!pip install setuptools_scm
!mkdir ./tmp
!CCACHE_DIR=./tmp python setup.py develop

# Install Packages for Fine-Tuning

In [ ]:
!pip install -r bitsandbytes transformers datasets trl peft accelerate

# Fine-Tuning Code (for Benchmarking Purposes)

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)

compute_dtype = torch.bfloat16
attn_implementation = 'flash_attention_2'

def fine_tune(model_name, auto_find_batch_size, batch_size=1, gradient_accumulation_steps=32):

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = "<|im_end|>"
  tokenizer.pad_token_id = 151645
  tokenizer.padding_side = 'left'
  mname = model_name.split('/')[-1]
  ds = load_dataset("timdettmers/openassistant-guanaco")

  #Add the EOS token
  def process(row):
      row["text"] = row["text"]+tokenizer.eos_token
      return row

  ds = ds.map(
      process,
      num_proc= multiprocessing.cpu_count(),
      load_from_cache_file=False,
  )


  model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map={"": 0}, torch_dtype=compute_dtype#, attn_implementation=attn_implementation
  )
  model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})

  if auto_find_batch_size:
    batch_size = 1
    gradient_accumulation_steps = 1


  peft_config = LoraConfig(
          lora_alpha=16,
          lora_dropout=0.05,
          r=16,
          bias="none",
          task_type="CAUSAL_LM",
          target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"]
  )

  output_dir = "./LoRA-"+mname+"/"

  training_arguments = SFTConfig(
          output_dir=output_dir,
          optim="adamw_8bit",
          per_device_train_batch_size=batch_size,
          gradient_accumulation_steps=gradient_accumulation_steps,
          log_level="debug",
          save_strategy="no",
          logging_steps=25,
          learning_rate=1e-5,
          bf16 = True,
          max_steps=50,
          warmup_ratio=0.1,
          lr_scheduler_type="linear",
          dataset_text_field="text",
          max_seq_length=512,
          auto_find_batch_size=auto_find_batch_size,
          report_to="none"
  )

  trainer = SFTTrainer(
          model=model,
          train_dataset=ds['train'],
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
  )

  #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

  trainer_ = trainer.train()


  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
  print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
  print("-----")
  #----

## Batch size of 1

In [ ]:
fine_tune("Qwen/Qwen2.5-7B", False, batch_size=1, gradient_accumulation_steps=32)

Repo card metadata block was not found. Setting CardData to empty.


Map (num_proc=128):   0%|          | 0/9846 [00:00<?, ? examples/s]

Map (num_proc=128):   0%|          | 0/518 [00:00<?, ? examples/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text. If text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 9,846
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 50
  Number of trainable parameters = 40,370,176
`use_cache=True` is incompatible with gradient checkpointi

GPU = NVIDIA GeForce RTX 5090. Max memory = 31.367 GB.
14.363 GB of memory reserved.


Step,Training Loss
25,1.332000
50,1.312000




Training completed. Do not forget to share your model on huggingface.co/models =)




269.9567 seconds used for training.
4.5 minutes used for training.
Peak reserved memory = 15.852 GB.
Peak reserved memory for training = 1.489 GB.
Peak reserved memory % of max memory = 50.537 %.
Peak reserved memory for training % of max memory = 4.747 %.
-----


## Batch size of 8 (can run with 16)

In [ ]:
fine_tune("Qwen/Qwen2.5-7B", False, batch_size=8, gradient_accumulation_steps=4)

# Benchmarking vLLM

I had issues with the V1 engine so I deactivated it with VLLM_USE_V1=0. I would expect these issues to be fixed soon, so don't hesitate to try again without disabling the V1. It could be faster.

In [ ]:
!VLLM_USE_V1=0 python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct --num-prompts 1000 --input-len 512 --output-len 1024 --output-json 16bit-1000-512-1024.json

INFO 03-22 20:33:54 [metrics.py:481] Avg prompt throughput: 14720.4 tokens/s, Avg generation throughput: 27.1 tokens/s, Running: 256 reqs, Swapped: 0 reqs, Pending: 744 reqs, GPU KV cache usage: 79.2%, CPU KV cache usage: 0.0%.
INFO 03-22 20:33:59 [metrics.py:481] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 7202.7 tokens/s, Running: 256 reqs, Swapped: 0 reqs, Pending: 744 reqs, GPU KV cache usage: 99.5%, CPU KV cache usage: 0.0%.
WARNING 03-22 20:33:59 [scheduler.py:1769] Sequence group 255 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1
INFO 03-22 20:34:04 [metrics.py:481] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 6406.0 tokens/s, Running: 215 reqs, Swapped: 0 reqs, Pending: 785 reqs, GPU KV cache usage: 99.9%, CPU KV cache usage: 0.0%.
W

In [ ]:
!VLLM_USE_V1=0  python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --num-prompts 1000 --input-len 512 --output-len 1024 --output-json 4bitAWQ-1000-512-1024.json

INFO 03-22 20:41:21 [__init__.py:256] Automatically detected platform cuda.
When dataset path is not set, it will default to random dataset
Namespace(backend='vllm', dataset_name='random', dataset=None, dataset_path=None, input_len=512, output_len=1024, n=1, num_prompts=1000, hf_max_batch_size=None, output_json='4bitAWQ-1000-512-1024.json', async_engine=False, disable_frontend_multiprocessing=False, disable_detokenize=False, lora_path=None, prefix_len=None, random_range_ratio=None, hf_subset=None, hf_split=None, model='Qwen/Qwen2.5-7B-Instruct-AWQ', task='auto', tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', hf_config_path=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='auto', kv_cache_dtype='auto', max_model_len=None, guided_decoding_backend='xgrammar', logits_processor_pattern=No

In [ ]:
!VLLM_USE_V1=0  python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct --num-prompts 32 --input-len 512 --output-len 1024 --output-json 4bitAWQ-32-512-1024.json

INFO 03-22 20:38:31 [__init__.py:256] Automatically detected platform cuda.
When dataset path is not set, it will default to random dataset
Namespace(backend='vllm', dataset_name='random', dataset=None, dataset_path=None, input_len=512, output_len=1024, n=1, num_prompts=32, hf_max_batch_size=None, output_json='4bitAWQ-32-512-1024.json', async_engine=False, disable_frontend_multiprocessing=False, disable_detokenize=False, lora_path=None, prefix_len=None, random_range_ratio=None, hf_subset=None, hf_split=None, model='Qwen/Qwen2.5-7B-Instruct', task='auto', tokenizer='Qwen/Qwen2.5-7B-Instruct', hf_config_path=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='auto', kv_cache_dtype='auto', max_model_len=None, guided_decoding_backend='xgrammar', logits_processor_pattern=None, model_im

In [ ]:
!VLLM_USE_V1=0  python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --num-prompts 32 --input-len 512 --output-len 1024 --output-json 4bitAWQ-32-512-1024.json

INFO 03-22 20:46:22 [__init__.py:256] Automatically detected platform cuda.
When dataset path is not set, it will default to random dataset
Namespace(backend='vllm', dataset_name='random', dataset=None, dataset_path=None, input_len=512, output_len=1024, n=1, num_prompts=32, hf_max_batch_size=None, output_json='4bitAWQ-32-512-1024.json', async_engine=False, disable_frontend_multiprocessing=False, disable_detokenize=False, lora_path=None, prefix_len=None, random_range_ratio=None, hf_subset=None, hf_split=None, model='Qwen/Qwen2.5-7B-Instruct-AWQ', task='auto', tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', hf_config_path=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='auto', kv_cache_dtype='auto', max_model_len=None, guided_decoding_backend='xgrammar', logits_processor_pattern=None, 

In [ ]:
!VLLM_USE_V1=0  python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct --num-prompts 32 --input-len 1024 --output-len 2048 --output-json 16-bit-32-1024-2048.json

INFO 03-22 20:39:27 [__init__.py:256] Automatically detected platform cuda.
When dataset path is not set, it will default to random dataset
Namespace(backend='vllm', dataset_name='random', dataset=None, dataset_path=None, input_len=1024, output_len=2048, n=1, num_prompts=32, hf_max_batch_size=None, output_json='16-bit-32-1024-2048.json', async_engine=False, disable_frontend_multiprocessing=False, disable_detokenize=False, lora_path=None, prefix_len=None, random_range_ratio=None, hf_subset=None, hf_split=None, model='Qwen/Qwen2.5-7B-Instruct', task='auto', tokenizer='Qwen/Qwen2.5-7B-Instruct', hf_config_path=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='auto', kv_cache_dtype='auto', max_model_len=None, guided_decoding_backend='xgrammar', logits_processor_pattern=None, model_i

In [ ]:
!VLLM_USE_V1=0 python vllm/benchmarks/benchmark_throughput.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --num-prompts 32 --input-len 1024 --output-len 2048 --output-json 4bitAWQ-32-1024-2048.json

INFO 03-22 20:47:12 [__init__.py:256] Automatically detected platform cuda.
When dataset path is not set, it will default to random dataset
Namespace(backend='vllm', dataset_name='random', dataset=None, dataset_path=None, input_len=1024, output_len=2048, n=1, num_prompts=32, hf_max_batch_size=None, output_json='4bitAWQ-32-1024-2048.json', async_engine=False, disable_frontend_multiprocessing=False, disable_detokenize=False, lora_path=None, prefix_len=None, random_range_ratio=None, hf_subset=None, hf_split=None, model='Qwen/Qwen2.5-7B-Instruct-AWQ', task='auto', tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', hf_config_path=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='auto', kv_cache_dtype='auto', max_model_len=None, guided_decoding_backend='xgrammar', logits_processor_pattern=None